# OffTheBall · Google Colab GPU 분석
런타임에서 **런타임 유형 변경 → T4 GPU**를 선택한 뒤 위에서부터 실행하세요. 영상 한 개를 업로드하면 분석 영상과 좌표 기록을 ZIP으로 내려받습니다.

In [ ]:
REPOSITORY = 'https://github.com/wobeen/OffTheBall.git'
BRANCH = 'codex/colab'
IMAGE_SIZE = 960  # 메모리가 부족하면 640으로 낮추세요.

import os, shutil, subprocess, sys
from pathlib import Path
workspace = Path('/content/OffTheBall')
if workspace.exists():
    shutil.rmtree(workspace)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPOSITORY, str(workspace)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(workspace/'requirements-colab.txt')], check=True)
os.chdir(workspace)
sys.path.insert(0, str(workspace/'src'))
print('설치 완료')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU가 없습니다. 런타임 유형을 T4 GPU로 변경하세요.')
print('GPU:', torch.cuda.get_device_name(0))

from ultralytics import YOLO
model_dir = workspace/'models'
model_dir.mkdir(exist_ok=True)
downloaded = Path(YOLO('yolo11n.pt').ckpt_path)
target = model_dir/'yolo11n.pt'
if downloaded.resolve() != target.resolve():
    shutil.copy2(downloaded, target)
print('모델 준비:', target)

In [ ]:
from google.colab import files
USE_GOOGLE_DRIVE = False
DRIVE_VIDEO = '/content/drive/MyDrive/match.mp4'
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    video_path = Path(DRIVE_VIDEO)
    if not video_path.is_file():
        raise FileNotFoundError(video_path)
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('분석할 영상 파일 한 개만 업로드하세요.')
    video_name = next(iter(uploaded))
    video_path = workspace/video_name
print('분석 영상:', video_path.name)

In [ ]:
from offtheball.detection import PersonDetector
from offtheball.pipeline import analyze_video

result_dir = workspace/'colab_result'
if result_dir.exists():
    shutil.rmtree(result_dir)
detector = PersonDetector(device=0, image_size=IMAGE_SIZE)
output, summary = analyze_video(video_path, detector, output_dir=result_dir)
print(summary)

In [ ]:
archive = shutil.make_archive('/content/OffTheBall_result', 'zip', result_dir)
files.download(archive)
print('완료:', archive)

결과 ZIP에는 `analysis.mp4`, `frames.jsonl`, `summary.json`이 들어 있습니다. 코랩 세션은 종료되면 파일이 사라지므로 마지막 다운로드 셀까지 실행하세요.